# 프로젝트 1: PVD AlCu 데이터 이해하기

이 노트북은 분석에 앞서 데이터의 구성과 해석상 주의점을 정리합니다. 현재 단계에서는 코드, 전처리, 모델 학습 또는 분석 결론을 포함하지 않습니다.

## 1. 프로젝트 목표

AlCu 박막 증착 공정에서 설비 센서와 두께의 관계를 살펴보고, 평균 두께와 웨이퍼 내 산포를 설명할 수 있는 변수를 탐색합니다. 이후 회귀분석과 추가 실험 설계로 이어갑니다.

## 2. 데이터 출처와 파일

[Zenodo: Advanced Process Control and Statistical Process Control Data for Thickness Prediction of AlCu and WTi Metal Layer](https://zenodo.org/records/16881338)에서 공개한 실제 반도체 제조 PVD 공정 데이터 중 AlCu 파일을 사용합니다. 원 출처는 Infineon Technologies의 공정 데이터입니다.

| 파일 | 역할 | 확인한 크기 |
|---|---|---|
| `../datasets/pvd_alcu/X_pvd_AlCu.csv` | 공정 중 설비 센서 집계값 | 4,848행 × 97열 |
| `../datasets/pvd_alcu/Y_pvd_AlCu.csv` | 공정 후 17개 지점의 두께 측정값 | 4,848행 × 17열 |

행과 열의 개수는 이 저장소에 내려받은 CSV에서 헤더를 제외하고 확인했습니다. 경로는 노트북이 있는 폴더를 기준으로 표시했습니다. 데이터셋과 관련 논문의 인용 정보는 원본 페이지를 참고하세요.

## 3. 입력 데이터 X: 공정 센서

`Sensor_1`부터 `Sensor_97`까지 97개 열이 있습니다. 원본 설명에 따르면 가스 유량, 온도, 전압, 전류 등 장비 내부 센서 시계열을 집계한 값입니다. 공정 중 상태를 나타내는 입력 변수로 사용할 예정입니다.

**주의:** 열 이름이 익명화되어 있어 각 번호가 어떤 물리량인지, 단위가 무엇인지 이 파일만으로는 알 수 없습니다. 센서값은 측정된 공정 상태이며, 실제로 직접 조절 가능한 레시피 설정값인지 확인되지 않았습니다.

## 4. 출력 데이터 Y: 두께 측정

`RAW_VALUE_t_1`부터 `RAW_VALUE_t_17`까지 17개 열이 있습니다. 원본 설명상 각 열은 두께 측정 지점에 해당합니다. 한 행의 17개 값을 함께 살펴보면 해당 관측의 평균 두께와 지점 간 산포를 계산할 수 있습니다.

예를 들어 평균은 17개 값의 합을 17로 나눈 값이고, 표준편차는 그 값들이 평균에서 얼마나 퍼져 있는지 나타냅니다. Uniformity는 여러 정의가 가능하므로 실제 분석 전에 식을 정하고 명시해야 합니다.

측정값의 **물리 단위, 스케일 변환 여부, 각 지점의 웨이퍼상 위치**는 현재 확인되지 않았습니다. 값이 0과 1 사이로 보이더라도 정규화 방식이나 실제 두께 단위를 추정하면 안 됩니다. 공간 패턴이나 실제 규격과의 비교에는 추가 정보가 필요합니다.

## 5. X와 Y를 어떻게 함께 볼까?

X와 Y는 각각 4,848행으로 크기가 같습니다. 원본은 입력과 출력을 분리하여 제공하지만, CSV 열에는 명시적인 관측 ID가 없습니다. **행 수가 같다는 사실만으로 행 순서가 정확히 대응한다는 것이 증명되지는 않습니다.** 모델링 전에 원본 문서나 제공자 정보를 통해 매칭 기준과 각 행의 단위(예: 웨이퍼 또는 다른 생산 단위)를 확인해야 합니다.

대응이 확인되면 하나의 관측에 대해 센서 97개를 입력으로 두고, 17개 두께값 또는 그 요약 지표를 결과로 분석할 수 있습니다. 확인되지 않으면 행 순서 매칭은 가정임을 밝히고 결과 해석을 제한합니다.

## 6. 앞으로 확인할 질문

1. X와 Y의 행은 어떤 기준으로 연결되며 한 행은 무엇을 뜻하는가?
2. 결측값, 상수 열, 중복 행, 비정상적인 값이 있는가?
3. 17개 측정값의 단위와 변환 방식은 무엇인가? 지점 배치는 알려져 있는가?
4. 평균 두께와 산포는 어떤 센서 변수와 관련이 있는가? 변수끼리도 강한 상관이 있는가?
5. 시간, 설비, Lot, Recipe 정보가 제공되는가? 평가 시 데이터 누출을 어떻게 방지할 수 있는가?
6. 실제로 조절 가능한 변수와 실험군 정보가 있는가? 없다면 어떤 추가 Split 실험이 필요한가?

## 7. 해석 범위

이 데이터는 센서와 두께의 **관계와 예측 가능성**을 탐색하기에 적합합니다. 관측된 상관관계나 회귀계수만으로 변수 변경의 인과효과를 확정할 수는 없습니다. 계획된 Split 실험, 제어 가능한 조건과 실제 공정 규격이 확인되기 전에는 결과를 현장에 적용할 최적 레시피로 간주하지 않습니다.

다음 단계에서는 데이터 불러오기와 기초 품질 확인부터 시작합니다.